# **ТИПиС - ИДЗ№4 - Воропаев Илья 3391**



**Подготовка данных**

---



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.feature_selection import mutual_info_classif
import warnings
warnings.filterwarnings('ignore')

# Грузим данные
df = pd.read_csv('bank-full.csv', sep=';')

# Выбираем данные (необходимые признаки)
features = ['age', 'job', 'marital', 'education', 'balance', 'housing',
                    'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
                    'previous', 'poutcome', 'y']

df = df[features]
print("Итого данных:", df.shape)
print("Типы данных:\n",df.dtypes)
df.head().T

Итого данных: (45211, 15)
Типы данных:
 age           int64
job          object
marital      object
education    object
balance       int64
housing      object
contact      object
day           int64
month        object
duration      int64
campaign      int64
pdays         int64
previous      int64
poutcome     object
y            object
dtype: object


,0,1,2,3,4
age,58,44,33,47,33
job,management,technician,entrepreneur,blue-collar,unknown
marital,married,single,married,married,single
education,tertiary,secondary,secondary,unknown,unknown
balance,2143,29,2,1506,1
housing,yes,yes,yes,yes,no
contact,unknown,unknown,unknown,unknown,unknown
day,5,5,5,5,5
month,may,may,may,may,may
duration,261,151,76,92,198


In [ ]:
print("Количество пропущенных значений")
missing_values = df.isnull().sum()
print(missing_values)

Количество пропущенных значений
age          0
job          0
marital      0
education    0
balance      0
housing      0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64


**Вопрос 1 - Самое частое значение в столбце education**

---



In [ ]:
education_mode = df['education'].mode()[0]
print(df['education'].value_counts())

education
secondary    23202
tertiary     13301
primary       6851
unknown       1857
Name: count, dtype: int64


Ответ: самое частое значение - secondary

**Вопрос 2 - Наибольшая корреляция между числовыми признаками**

---

In [ ]:
# Выбираем только числовые признаки
numeric_features = ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
numeric_df = df[numeric_features]

# Создаем корреляционную матрицу
correlation_matrix = numeric_df.corr()

# Находим пару с наибольшей корреляцией
corr_pairs = []
for i in range(len(correlation_matrix.columns)):
    for j in range(i+1, len(correlation_matrix.columns)):
        col1 = correlation_matrix.columns[i]
        col2 = correlation_matrix.columns[j]
        corr_value = abs(correlation_matrix.iloc[i, j])
        corr_pairs.append((col1, col2, corr_value))

# Сортируем по убыванию корреляции
corr_pairs_sorted = sorted(corr_pairs, key=lambda x: x[2], reverse=True)

# Выводим данные
print("Корреляционная матрица:")
print(correlation_matrix)
print("\nКорреляция пар:")
for pair in corr_pairs_sorted[:5]:
    print(f"{pair[0]} и {pair[1]}: {pair[2]:.6f}")
print("Два признака, имеющие наибольшую корреляцию:")
print(corr_pairs_sorted[0][0], corr_pairs_sorted[0][1])

Корреляционная матрица:
               age   balance       day  duration  campaign     pdays  previous
age       1.000000  0.097783 -0.009120 -0.004648  0.004760 -0.023758  0.001288
balance   0.097783  1.000000  0.004503  0.021560 -0.014578  0.003435  0.016674
day      -0.009120  0.004503  1.000000 -0.030206  0.162490 -0.093044 -0.051710
duration -0.004648  0.021560 -0.030206  1.000000 -0.084570 -0.001565  0.001203
campaign  0.004760 -0.014578  0.162490 -0.084570  1.000000 -0.088628 -0.032855
pdays    -0.023758  0.003435 -0.093044 -0.001565 -0.088628  1.000000  0.454820
previous  0.001288  0.016674 -0.051710  0.001203 -0.032855  0.454820  1.000000

Корреляция пар:
pdays и previous: 0.454820
day и campaign: 0.162490
age и balance: 0.097783
day и pdays: 0.093044
campaign и pdays: 0.088628
Два признака, имеющие наибольшую корреляцию:
pdays previous


Ответ: pdays и previous имеют наибольшую корреляцию

In [ ]:
# Кодирование целевой переменной
df['y']=(df['y'] == 'yes').astype(int)

In [ ]:
# Разделяем на train+val и test (60%+20% и 20%)
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Затем разделяем train+val на train и val (60% и 20% от исходных)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

print(f"Тренировочный набор: {X_train.shape[0]} записей ({X_train.shape[0]/len(X)*100:.1f}%)")
print(f"Валидационный набор: {X_val.shape[0]} записей ({X_val.shape[0]/len(X)*100:.1f}%)")
print(f"Тестовый набор: {X_test.shape[0]} записей ({X_test.shape[0]/len(X)*100:.1f}%)")

Тренировочный набор: 27126 записей (60.0%)
Валидационный набор: 9042 записей (20.0%)
Тестовый набор: 9043 записей (20.0%)


**Вопрос 3 - Наибольшая взаимная информация**

---

In [ ]:
# Выбираем категориальные признаки
categorical_features = ['job', 'marital', 'education', 'housing', 'contact', 'month', 'poutcome']

# Кодируем категориальные признаки для расчета взаимной информации
X_encoded = X_train[categorical_features].copy()
for col in categorical_features:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X_train[col])
mi_scores = mutual_info_classif(X_encoded, y_train, random_state=42)
mi_rounded = {feat: round(score, 2) for feat, score in zip(categorical_features, mi_scores)}
best_feature = max(mi_rounded, key=mi_rounded.get)
print(best_feature)

poutcome


Ответ: poutcome имеет наибольшую взаимную информацию

**Вопрос 4 - Точность логистической регрессии**

---

In [ ]:
categorical_features = ['job', 'marital', 'education', 'housing', 'contact', 'month', 'poutcome']

# One-hot кодирование категориальных переменных

# Для обучающей выборки
X_train_encoded = pd.get_dummies(X_train, columns=categorical_features, drop_first=True)

# Для валидационной выборки - важно использовать те же самые фичи!
X_val_encoded = pd.get_dummies(X_val, columns=categorical_features, drop_first=True)

# Выравнивание признаков между train и val
train_cols = set(X_train_encoded.columns)
val_cols = set(X_val_encoded.columns)

# Находим общие признаки
common_cols = list(train_cols.intersection(val_cols))

# Применяем только общие признаки
X_train_final = X_train_encoded[common_cols]
X_val_final = X_val_encoded[common_cols]
print(f"Финальный размер обучающей выборки: {X_train_final.shape}")
print(f"Финальный размер валидационной выборки: {X_val_final.shape}")

# Обучение логистической регрессии с указанными параметрами
model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)
model.fit(X_train_final, y_train)

# Предсказание на валидационной выборке
y_val_pred = model.predict(X_val_final)

# Расчет точности
accuracy = accuracy_score(y_val, y_val_pred)
accuracy_rounded = round(accuracy, 2)
print(f"Точность на валидационной выборке: {accuracy:.4f}")
print(f"Точность, округленная до двух знаков: {accuracy_rounded}")

Финальный размер обучающей выборки: (27126, 40)
Финальный размер валидационной выборки: (9042, 40)
Точность на валидационной выборке: 0.9035
Точность, округленная до двух знаков: 0.9


Ответ: 0.9

**Вопрос 5 - Наименее полезный признак**

---

In [ ]:
features = ['age', 'balance', 'marital', 'previous']
accuracy_differences = {}

for feature in features:
    # Создаем копии данных без одного признака
    if feature in categorical_features:
        # Для категориальных признаков исключаем все one-hot колонки
        cols_to_drop = [col for col in X_train_encoded.columns if col.startswith(feature + '_')]
        X_train_reduced = X_train_encoded.drop(cols_to_drop, axis=1)
        X_val_reduced = X_val_encoded.drop(cols_to_drop, axis=1)
    else:
        # Для числовых признаков
        X_train_reduced = X_train_encoded.drop(feature, axis=1)
        X_val_reduced = X_val_encoded.drop(feature, axis=1)

    # Обучаем модель без признака
    model_reduced = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
    )
    model_reduced.fit(X_train_reduced, y_train)

    # Предсказания
    y_val_pred_reduced = model_reduced.predict(X_val_reduced)
    accuracy_reduced = accuracy_score(y_val, y_val_pred_reduced)

    # Разница в точности
    difference = accuracy - accuracy_reduced
    accuracy_differences[feature] = difference

# Находим признак с наименьшей разницей
min_diff_feature = min(accuracy_differences, key=lambda x: abs(accuracy_differences[x]))
print(min_diff_feature)

balance


Ответ: balance

**Вопрос 6 - Лучшее значение параметра C**

---

In [73]:
C_values = [0.01, 0.1, 1, 10, 100]

# Обучаем модель с разными параметрами
for C in C_values:

    print(f"\nОбучение модели с C = {C}")

    # Создаем и обучаем модель с указанным C
    model = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train_final, y_train)

    # Предсказание на валидационной выборке
    y_val_pred = model.predict(X_val_final)

    # Расчет точности с округлением до трех знаков
    accuracy = accuracy_score(y_val, y_val_pred)
    accuracy_rounded = round(accuracy, 3)

    print(f"  Точность на валидационном наборе: {accuracy:.4f}")
    print(f"  Точность (округленная до 3 знаков): {accuracy_rounded}")


Обучение модели с C = 0.01
  Точность на валидационном наборе: 0.8987
  Точность (округленная до 3 знаков): 0.899

Обучение модели с C = 0.1
  Точность на валидационном наборе: 0.9025
  Точность (округленная до 3 знаков): 0.902

Обучение модели с C = 1
  Точность на валидационном наборе: 0.9035
  Точность (округленная до 3 знаков): 0.903

Обучение модели с C = 10
  Точность на валидационном наборе: 0.9031
  Точность (округленная до 3 знаков): 0.903

Обучение модели с C = 100
  Точность на валидационном наборе: 0.9031
  Точность (округленная до 3 знаков): 0.903


Ответ: к наилучшей точности на валидационном наборе приводит (0.9035) параметр С = 1

**Итоговые ответы**

---

1. самое частое значение - secondary
2. pdays и previous имеют наибольшую корреляцию
3. poutcome имеет наибольшую взаимную информацию
4. получил точность 0.9
5. balance имеет наименьшую разницу
6. к наилучшей точности на валидационном наборе приводит (0.9035) параметр С = 1